In [1]:
# Load 
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# Add project root to path so we can import from `generation/`
sys.path.insert(0, str(Path.cwd().parent))

import torch
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
torch.cuda.empty_cache()
pipe.to("cuda")
print("Loaded pipe!")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pipe!


In [2]:
from LatentPredictionDataset import Latent
def get_latent_lists_from_seeds(seeds: list[int]) -> list[Latent]:
    generators = [torch.Generator(device="cpu").manual_seed(seed) for seed in seeds]
    latents = [
            pipe.prepare_latents(
                batch_size=1,
                num_channels_latents=pipe.unet.config.in_channels,
                height=512,
                width=512,
                dtype=pipe.unet.dtype,
                device="cpu",
                generator=generator,
            )[0] / pipe.scheduler.init_noise_sigma
            for generator in generators
        ]
    return latents

In [21]:
import pandas as pd
from torch.utils.data import DataLoader
from LatentPredictionDataset import create_splits, LatentPredictionDataset

full_dataframe = pd.read_csv("../results/baseline_metrics.csv")
print(f"Loaded {len(full_dataframe)} rows")

# 1. Create the split DataFrames
train_df, val_df = create_splits(full_dataframe)
print(f"Train df Rows: {len(train_df)} | Val dfRows: {len(val_df)}")

# 2. Create two separate Dataset instances
# (They don't know about each other, they just see their own data)
train_dataset = LatentPredictionDataset(
    metrics_df=train_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9),
    dtype=torch.float32
)

val_dataset = LatentPredictionDataset(
    metrics_df=val_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9),
    dtype=torch.float32
)

print(f"Train dataset length: {len(train_dataset)} | Val dataset length: {len(val_dataset)}")

Loaded 9600 rows
Total Groups: 600
Train Rows: 8640 | Val Rows: 960
Train df Rows: 8640 | Val dfRows: 960
Train dataset length: 1095 | Val dataset length: 105


# Train latent Predictor

In [4]:
from training.LatentPredictor import LatentShiftNetwork
latent_predictor = LatentShiftNetwork()
print(latent_predictor)

example = train_dataset[1]

example_prediction = latent_predictor(example["input_winner_latent"], example["input_loser_latents"])

LatentShiftNetwork(
  (net): Sequential(
    (0): Conv2d(8, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Conv2d(64, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)
x:  torch.Size([1, 8, 64, 64]) torch.float32 cpu
delta:  torch.Size([1, 4, 64, 64]) torch.float32 cpu


In [ ]:
from training.LatentPredictorTrainer import LatentShiftTrainer, LatentTrainerConfig

BATCH_SIZE = 64

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False) # No shuffle for Val usually")

# 1. Configure hyperparameters
# Start with modest settings. If directionality is low, increase lambda_triplet.
config = LatentTrainerConfig(
    lr=1e-4, 
    epochs=15, 
    triplet_weight=0.0,   # Scale the ranking loss. For now, only MSE on the winner latent
    triplet_margin=1.0,   # Ensure significant separation
    device="cuda"
)

# 2. Initialize Trainer with your existing model and loaders
trainer = LatentShiftTrainer(
    model=latent_predictor, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    config=config
)

# 3. Train
trainer.fit()

Starting training on cuda
Config: MSE + (0.0 * TripletLoss)


Training:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 01 | Train: 0.2113 | Val MSE: 0.2032 | Val Direction: 0.0376


Training:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 02 | Train: 0.2065 | Val MSE: 0.1920 | Val Direction: 0.0425


Training:   0%|          | 0/18 [00:00<?, ?it/s]